### Homework 
#### Please download DEM in Pennsylvania and run the HAND model

- Please download the DEM tiles for Pennsylvania
- Run the HAND model to estimate the inudation depth for each tile
- Export the results to geotiff
- Mosaic the exposed results into a complete `geotiff` for whole Pennsylvania.
- Upload your notebook and submit a report to present your map for PA. 

Here is a hint about how to conduct the analysis for one tile. You can reference this for your homework. You need to generate the results for whole Pennsylvania. You will need to use loop to download DEM tiles for Pennsylvania, and conduct the mosaicing to generate the image for the whole state. 

Use the download-dem.ipynb to download the dem data for any location. 


### Download Pennsylvania DEM tiles from USGS

In [25]:
from pathlib import Path
from urllib.parse import urlparse, unquote
import json
import time
import re
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

PA_BBOX = (-80.53, 39.71, -74.68, 42.52)
DATASET = "National Elevation Dataset (NED) 1 arc-second Current"
API_URL = "https://tnmaccess.nationalmap.gov/api/v1/products"

assignment_dir = Path("assignment/assignment_2")
if not assignment_dir.is_dir():
    assignment_dir = Path.cwd()
DEM_DIR = assignment_dir / "DEM_Tiles"
DEM_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
retry = Retry(total=3, backoff_factor=1,
              status_forcelist=[429, 500, 502, 503, 504],
              allowed_methods=["GET"])
session.mount("https://", HTTPAdapter(max_retries=retry))


def find_pa_dem_products():
    products = {}
    offset = 0
    while True:
        response = session.get(API_URL, params={
            "bbox": ",".join(map(str, PA_BBOX)),
            "datasets": DATASET,
            "prodFormats": "GeoTIFF",
            "max": 100,
            "offset": offset,
        }, timeout=(15, 120))
        response.raise_for_status()
        payload = response.json()
        if payload.get("error") or payload.get("errors"):
            raise RuntimeError(f"USGS query failed: {payload}")
        items = payload.get("items", [])

        previous_count = len(products)
        for item in items:
            url = item.get("downloadURL", "")
            products[url] = item
        offset += len(items)
        if offset >= int(payload.get("total", offset)):
            break

    latest = {}
    def edition_rank(item):
        url = item["downloadURL"]
        dates = re.findall(r"_(\d{8})(?=\.tif)", url)
        return ("/current/" in url, dates[-1] if dates else "",
                item.get("publicationDate") or "")
    for item in products.values():
        match = re.search(r"USGS_1_([ns]\d{2}[ew]\d{3})(?:_|\.)", item["downloadURL"])
        tile = match.group(1)
        if tile not in latest or edition_rank(item) > edition_rank(latest[tile]):
            latest[tile] = item
    return [latest[tile] for tile in sorted(latest)]


products = find_pa_dem_products()
(DEM_DIR / "pa_dem_products.json").write_text(
    json.dumps(products, indent=2), encoding="utf-8")
print(f"Found {len(products)} DEM files (~30 m). Destination: {DEM_DIR.resolve()}")
for item in products:
    print(item.get("title", item["downloadURL"]))


Found 28 DEM files (~30 m). Destination: /Users/xuanyuzhou/Desktop/MUSA 6950 AI for Urban Sustainability/ai-urban-sustainability/assignment/assignment_2/DEM_Tiles
USGS 1 Arc Second n40w075 20210624
USGS 1 Arc Second n40w076 20250507
USGS 1 Arc Second n40w077 20260512
USGS 1 Arc Second n40w078 20260512
USGS 1 Arc Second n40w079 20260512
USGS 1 Arc Second n40w080 20260512
USGS 1 Arc Second n40w081 20230911
USGS 1 Arc Second n41w075 20221115
USGS 1 Arc Second n41w076 20221115
USGS 1 Arc Second n41w077 20220429
USGS 1 Arc Second n41w078 20220429
USGS 1 Arc Second n41w079 20220429
USGS 1 Arc Second n41w080 20210630
USGS 1 Arc Second n41w081 20230911
USGS 1 Arc Second n42w075 20240925
USGS 1 Arc Second n42w076 20230227
USGS 1 Arc Second n42w077 20230227
USGS 1 Arc Second n42w078 20230227
USGS 1 Arc Second n42w079 20220429
USGS 1 Arc Second n42w080 20220429
USGS 1 Arc Second n42w081 20230911
USGS 1 Arc Second n43w075 20241010
USGS 1 Arc Second n43w076 20230227
USGS 1 Arc Second n43w077 202302

In [ ]:
def download_dem(product):
    url = product["downloadURL"]
    filename = Path(unquote(urlparse(url).path)).name
    destination = DEM_DIR / filename
    marker = DEM_DIR / (filename + ".complete.json")
    temporary = DEM_DIR / (filename + ".part")

    if destination.exists() and marker.exists():
        saved = json.loads(marker.read_text(encoding="utf-8"))
        if saved.get("url") == url and saved.get("bytes") == destination.stat().st_size:
            print(f"Skip completed: {filename}")
            return destination

    for attempt in range(3):
        try:
            with session.get(url, stream=True, timeout=(15, 120),
                             headers={"Accept-Encoding": "identity"}) as response:
                response.raise_for_status()
                expected = int(response.headers.get("Content-Length", 0))
                with temporary.open("wb") as output:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            output.write(chunk)
                size = temporary.stat().st_size
                if size == 0 or (expected and size != expected):
                    raise IOError(f"Incomplete download: {size} of {expected} bytes")
                with temporary.open("rb") as source:
                    signature = source.read(4)
                if signature not in (b"II\x2a\x00", b"MM\x00\x2a",
                                     b"II\x2b\x00", b"MM\x00\x2b"):
                    raise IOError("Response is not a TIFF file")
            temporary.replace(destination)
            marker.write_text(json.dumps({"url": url, "bytes": size}), encoding="utf-8")
            print(f"Downloaded: {filename} ({size / 1024**2:.1f} MB)")
            return destination
        except (requests.RequestException, OSError) as exc:
            temporary.unlink(missing_ok=True)
            if attempt == 2:
                raise
            print(f"Retry {attempt + 1}/2: {filename}: {exc}")
            time.sleep(2 ** (attempt + 1))


dem_files = []
failed = []
for index, product in enumerate(products, start=1):
    print(f"[{index}/{len(products)}]", end=" ")
    try:
        dem_files.append(download_dem(product))
    except (requests.RequestException, OSError) as exc:
        failed.append(product["downloadURL"])
        print(f"FAILED: {exc}")


[1/28] Downloaded: USGS_1_n40w075_20210624.tif (37.6 MB)
[2/28] Downloaded: USGS_1_n40w076_20250507.tif (53.5 MB)
[3/28] Downloaded: USGS_1_n40w077_20260512.tif (53.1 MB)
[4/28] Downloaded: USGS_1_n40w078_20260512.tif (55.9 MB)
[5/28] Downloaded: USGS_1_n40w079_20260512.tif (56.1 MB)
[6/28] Downloaded: USGS_1_n40w080_20260512.tif (53.6 MB)
[7/28] Downloaded: USGS_1_n40w081_20230911.tif (56.8 MB)
[8/28] Downloaded: USGS_1_n41w075_20221115.tif (57.0 MB)
[9/28] Downloaded: USGS_1_n41w076_20221115.tif (56.0 MB)
[10/28] Downloaded: USGS_1_n41w077_20220429.tif (54.9 MB)
[11/28] Downloaded: USGS_1_n41w078_20220429.tif (55.0 MB)
[12/28] Downloaded: USGS_1_n41w079_20220429.tif (53.6 MB)
[13/28] Downloaded: USGS_1_n41w080_20210630.tif (54.6 MB)
[14/28] Downloaded: USGS_1_n41w081_20230911.tif (54.7 MB)
[15/28] Downloaded: USGS_1_n42w075_20240925.tif (54.8 MB)
[16/28] Downloaded: USGS_1_n42w076_20230227.tif (53.1 MB)
[17/28] Downloaded: USGS_1_n42w077_20230227.tif (54.0 MB)
[18/28] Downloaded: USG

### flood

In [3]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from pysheds.grid import Grid
import pysheds
import numpy as np


In [23]:
from pathlib import Path
import numpy as np
from pysheds.grid import Grid


def process_dem_tile(
    dem_path,
    output_dir,
    channel_threshold=200,
    water_level=5.0
):
    dem_path = Path(dem_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    grid = Grid.from_raster(str(dem_path))
    dem = grid.read_raster(str(dem_path))

    valid = np.isfinite(dem)
    if dem.nodata is not None:
        valid &= dem != dem.nodata

    dem_for_flow = dem.astype(np.float64).copy()
    dem_for_flow[valid] = np.round(dem_for_flow[valid], 3)

    pit_filled_dem = grid.fill_pits(dem_for_flow)
    flooded_dem = grid.fill_depressions(pit_filled_dem)
    inflated_dem = grid.resolve_flats(flooded_dem)

    fdir = grid.flowdir(inflated_dem)
    acc = grid.accumulation(fdir)

    channels = (acc > channel_threshold) & valid

    hand = grid.compute_hand(fdir, dem, channels)

    hand_values = np.asarray(hand)
    usable = valid & np.isfinite(hand_values)

    if hand.nodata is not None:
        usable &= hand_values != hand.nodata

    usable &= hand_values >= 0

    nodata_out = -9999.0
    depth = hand.astype(np.float64).copy()
    depth[:] = nodata_out
    depth[usable] = np.maximum(
        water_level - hand_values[usable], 0.0
    )

    output_path = output_dir / f"{dem_path.stem}_depth.tif"

    grid.to_raster(
        depth,
        str(output_path),
        nodata=nodata_out,
        dtype=np.float32
    )

    return output_path

In [24]:
output_dir = DEM_DIR.parent / "inundation_tiles"

result_files = []
failed_tiles = []

for i, dem_path in enumerate(dem_files, start=1):
    print(f"\n[{i}/{len(dem_files)}] {dem_path.name}")

    try:
        output_path = process_dem_tile(
            dem_path,
            output_dir,
            channel_threshold=200,
            water_level=5.0
        )

        result_files.append(output_path)
        print(f"  save as:{output_path.name}")

    except Exception as exc:
        failed_tiles.append(dem_path)
        print(f"  failed: {type(exc).__name__}: {exc}")



[1/28] USGS_1_n40w075_20210624.tif
  save as:USGS_1_n40w075_20210624_depth.tif

[2/28] USGS_1_n40w076_20250507.tif
  save as:USGS_1_n40w076_20250507_depth.tif

[3/28] USGS_1_n40w077_20260512.tif
  save as:USGS_1_n40w077_20260512_depth.tif

[4/28] USGS_1_n40w078_20260512.tif
  save as:USGS_1_n40w078_20260512_depth.tif

[5/28] USGS_1_n40w079_20260512.tif
  save as:USGS_1_n40w079_20260512_depth.tif

[6/28] USGS_1_n40w080_20260512.tif
  save as:USGS_1_n40w080_20260512_depth.tif

[7/28] USGS_1_n40w081_20230911.tif
  save as:USGS_1_n40w081_20230911_depth.tif

[8/28] USGS_1_n41w075_20221115.tif
  save as:USGS_1_n41w075_20221115_depth.tif

[9/28] USGS_1_n41w076_20221115.tif
  save as:USGS_1_n41w076_20221115_depth.tif

[10/28] USGS_1_n41w077_20220429.tif
  save as:USGS_1_n41w077_20220429_depth.tif

[11/28] USGS_1_n41w078_20220429.tif
  save as:USGS_1_n41w078_20220429_depth.tif

[12/28] USGS_1_n41w079_20220429.tif
  save as:USGS_1_n41w079_20220429_depth.tif

[13/28] USGS_1_n41w080_20210630.tif


In [26]:
from pathlib import Path
from contextlib import ExitStack
import rasterio
from rasterio.merge import merge

assignment_dir = Path(DEM_DIR).parent
input_dir = assignment_dir / "inundation_tiles"
output_file = assignment_dir / "PA_inundation_depth.tif"

dem_paths = sorted(Path(DEM_DIR).glob("*.tif"))


depth_paths = [
    input_dir / f"{dem_path.stem}_depth.tif"
    for dem_path in dem_paths
]

missing = [path.name for path in depth_paths if not path.exists()]

with ExitStack() as stack:
    sources = [
        stack.enter_context(rasterio.open(path))
        for path in depth_paths
    ]

    reference = sources[0]

    mosaic, out_transform = merge(
        sources,
        nodata=-9999.0,
        dtype="float32",
        method="first"
    )

    profile = reference.profile.copy()
    profile.update(
        driver="GTiff",
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=out_transform,
        count=1,
        dtype="float32",
        nodata=-9999.0,
        compress="lzw",
        BIGTIFF="IF_SAFER"
    )

    with rasterio.open(output_file, "w", **profile) as dst:
        dst.write(mosaic)
        dst.set_band_description(1, "Inundation depth (m)")
        dst.update_tags(
            water_level_m=5.0,
            channel_threshold_cells=200
        )

del mosaic

In [27]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask

boundary_file = assignment_dir / "Pennsylvania_State_Boundary.geojson"
mosaic_file = assignment_dir / "PA_inundation_depth.tif"
clipped_file = assignment_dir / "PA_inundation_depth_clipped.tif"

boundary = gpd.read_file(boundary_file)

with rasterio.open(mosaic_file) as src:
    boundary = boundary.to_crs(src.crs)

    clipped, clipped_transform = mask(
        src,
        boundary.geometry,
        crop=True,
        nodata=-9999.0
    )

    profile = src.profile.copy()
    profile.update(
        height=clipped.shape[1],
        width=clipped.shape[2],
        transform=clipped_transform,
        nodata=-9999.0,
        compress="lzw",
        BIGTIFF="IF_SAFER"
    )

    with rasterio.open(clipped_file, "w", **profile) as dst:
        dst.write(clipped)
        dst.set_band_description(1, "Inundation depth (m)")
        dst.update_tags(**src.tags())

del clipped